<a href="https://colab.research.google.com/github/jk98190/Bjj-App/blob/main/Bjj_app_Updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sys
!{sys.executable} -m pip install streamlit
!{sys.executable} -m pip install youtube-transcript-api pandas plotly google-generativeai

import streamlit as st
import pandas as pd
import datetime
import os
import re
import plotly.express as px
from youtube_transcript_api import YouTubeTranscriptApi

# AI SDK imports (Google GenAI)
try:
    from google import genai
    GENAI_AVAILABLE = True
except ImportError:
    GENAI_AVAILABLE = False

# ---------------------------------------------------------
# 1. SETUP & DATA INITIALIZATION
# ---------------------------------------------------------
st.set_page_config(page_title="BJJ Tracker & Study Bank", page_icon="🥋", layout="wide")

DATA_DIR = "bjj_master_data"
LOG_FILE = os.path.join(DATA_DIR, "training_logs.csv")
STUDY_FILE = os.path.join(DATA_DIR, "study_bank.csv")

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

if not os.path.exists(LOG_FILE):
    pd.DataFrame(columns=[
        "Date", "Rank", "Stripes", "Style", "Duration_Min",
        "Rounds", "Techniques_Trained", "Submissions_Given",
        "Submissions_Received", "Notes"
    ]).to_csv(LOG_FILE, index=False)

if not os.path.exists(STUDY_FILE):
    pd.DataFrame([
        {
            "Title": "Kimura Reaction Chain",
            "Category": "Guard / Closed Guard",
            "Athlete_Source": "Adele Fornarino System",
            "Video_URL": "https://www.youtube.com/watch?v=v1wTVPUS4I8",
            "Key_Details": "Kimura grip -> posture break -> collar tie. Branch A: Yank arm -> Clamp guard -> Triangle. Branch B: Duck hand -> Frame head -> Omoplata."
        },
        {
            "Title": "Half Guard Rescue: Knee Lever vs. Windmill",
            "Category": "Half Guard / Escapes",
            "Athlete_Source": "Bodoni / Glick System",
            "Video_URL": "https://www.youtube.com/watch?v=wg21Vd0Dzz8",
            "Key_Details": "Intercept crossface with bicep block. If knee on floor: Knee-elbow escape x2 -> Clamp. If knee raised: Windmill bridge + palm-up wrist grip -> Clamp."
        }
    ]).to_csv(STUDY_FILE, index=False)

# ---------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------
def extract_youtube_id(url):
    match = re.search(r"(?:v=|/)([0-9A-Za-z_-]{11})", url)
    return match.group(1) if match else None

def fetch_youtube_transcript(video_id):
    try:
        ytt = YouTubeTranscriptApi()
        transcript = ytt.fetch(video_id, languages=['en'])
        return " ".join([item['text'] for item in transcript])
    except Exception as e:
        return None

def analyze_transcript_to_tree(transcript, api_key):
    if not GENAI_AVAILABLE:
        return "Error: google-genai library is not installed."

    client = genai.Client(api_key=api_key)
    prompt = f"""
    You are an expert Brazilian Jiu-Jitsu black belt analyst.
    Analyze the following video transcript of a grappling technique or match study:

    "{transcript}"

    Provide a structured output containing:
    1. **Technique Summary & Core Principles** (3-4 bullet points)
    2. **Decision Tree Breakdown** formatted as a Markdown ASCII tree showing opponent reactions and counter-branches.

    Example ASCII Format:
    POSITION / ENTRY
       │
       ├──► [Opponent Reaction A] ➔ Counter Attack A
       └──► [Opponent Reaction B] ➔ Counter Attack B
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

# ---------------------------------------------------------
# APP HEADER & TABS
# ---------------------------------------------------------
st.title("🥋 BJJ Progression Tracker & Decision Tree Analyzer")

tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "📝 Log Session",
    "📚 Study Bank",
    "🤖 AI Video Breakdown",
    "🌳 System Decision Trees",
    "📊 Analytics"
])

# ---------------------------------------------------------
# TAB 1: LOG SESSION
# ---------------------------------------------------------
with tab1:
    st.header("Track Training Session")
    with st.form("log_form", clear_on_submit=True):
        col1, col2, col3 = st.columns(3)
        with col1:
            date_val = st.date_input("Date", datetime.date.today())
            rank_val = st.selectbox("Rank", ["White Belt", "Blue Belt", "Purple Belt", "Brown Belt", "Black Belt"])
        with col2:
            style_val = st.selectbox("Style", ["No-Gi", "Gi", "Open Mat", "Wrestling", "Private"])
            stripes_val = st.selectbox("Stripes", [0, 1, 2, 3, 4])
        with col3:
            duration_val = st.number_input("Duration (Minutes)", value=60, step=15)
            rounds_val = st.number_input("Rounds Sparred", value=5, step=1)

        tech_val = st.text_area("Techniques Trained", placeholder="e.g., Clamp guard entries, bicep block, pendulum sweep")
        col_s1, col_s2 = st.columns(2)
        with col_s1:
            subs_g_val = st.text_input("Submissions Given", placeholder="e.g., 1x RNC, 1x Triangle")
        with col_s2:
            subs_r_val = st.text_input("Submissions Received", placeholder="e.g., 1x Heel Hook")

        notes_val = st.text_area("Notes", placeholder="Physical fatigue, partner feedback, tactical timing...")

        submitted = st.form_submit_button("Save Session Log")
        if submitted:
            new_entry = {
                "Date": date_val.strftime('%Y-%m-%d'),
                "Rank": rank_val,
                "Stripes": stripes_val,
                "Style": style_val,
                "Duration_Min": duration_val,
                "Rounds": rounds_val,
                "Techniques_Trained": tech_val,
                "Submissions_Given": subs_g_val,
                "Submissions_Received": subs_r_val,
                "Notes": notes_val
            }
            df = pd.read_csv(LOG_FILE)
            df = pd.concat([df, pd.DataFrame([new_entry])], ignore_index=True)
            df.to_csv(LOG_FILE, index=False)
            st.success("✅ Training session saved successfully!")

# ---------------------------------------------------------
# TAB 2: STUDY BANK (Streamlit Native Widgets)
# ---------------------------------------------------------
with tab2:
    st.header("📚 Technique & Match Breakdown Library")

    # Form to add new study entry
    with st.expander("➕ Add New Study Breakdown Note", expanded=False):
        with st.form("study_form", clear_on_submit=True):
            col_a, col_b = st.columns(2)
            with col_a:
                s_title = st.text_input("Title", placeholder="e.g., Clamp Guard to Armbar")
                s_cat = st.selectbox("Category", ['Guard / Closed Guard', 'Half Guard / Escapes', 'Guard Passing', 'Wrestling / Takedowns', 'Submissions'])
            with col_b:
                s_source = st.text_input("Source / Athlete", placeholder="e.g., Adele Fornarino / Bodoni")
                s_url = st.text_input("YouTube URL", placeholder="https://www.youtube.com/watch?v=...")

            s_details = st.text_area("Key Details / Sequence Mechanics")
            if st.form_submit_button("Save Study Note"):
                new_study = {
                    "Title": s_title,
                    "Category": s_cat,
                    "Athlete_Source": s_source,
                    "Video_URL": s_url,
                    "Key_Details": s_details
                }
                df_s = pd.read_csv(STUDY_FILE)
                df_s = pd.concat([df_s, pd.DataFrame([new_study])], ignore_index=True)
                df_s.to_csv(STUDY_FILE, index=False)
                st.success("📖 Study note added!")

    # Display existing cards
    df_study = pd.read_csv(STUDY_FILE)
    st.subheader(f"Saved Study Cards ({len(df_study)})")

    for idx, row in df_study.iterrows():
        with st.container(border=True):
            st.markdown(f"### {row['Title']}")
            st.caption(f"**Category:** {row['Category']} | **Source:** {row['Athlete_Source']}")
            st.markdown(f"**Key Details:** {row['Key_Details']}")

            if pd.notna(row['Video_URL']) and "youtube.com" in str(row['Video_URL']):
                st.video(row['Video_URL'])

# ---------------------------------------------------------
# TAB 3: AI VIDEO TO DECISION TREE BREAKDOWN
# ---------------------------------------------------------
with tab3:
    st.header("🤖 Auto Video-to-Decision Tree Generator")
    st.write("Paste a Jiu-Jitsu video URL (with captions enabled) to automatically extract mechanics and generate a decision tree.")

    gemini_key = st.text_input("Google Gemini API Key", type="password", help="Enter your Gemini API key to run transcript analysis.")
    video_url_input = st.text_input("YouTube Video URL", placeholder="https://www.youtube.com/watch?v=...")

    if st.button("Generate Decision Tree Breakdown", type="primary"):
        if not gemini_key:
            st.error("Please provide a valid Gemini API Key.")
        elif not video_url_input:
            st.error("Please enter a YouTube video URL.")
        else:
            v_id = extract_youtube_id(video_url_input)
            if not v_id:
                st.error("Invalid YouTube URL.")
            else:
                with st.spinner("Fetching transcript from YouTube..."):
                    transcript = fetch_youtube_transcript(v_id)

                if not transcript:
                    st.error("Could not fetch transcripts for this video. Make sure the video has closed captions/subtitles enabled.")
                else:
                    st.success("Transcript fetched successfully!")
                    st.video(video_url_input)

                    with st.spinner("AI is analyzing grappling mechanics & building decision tree..."):
                        tree_analysis = analyze_transcript_to_tree(transcript, gemini_key)

                    st.markdown("### 📊 Generated Analysis & Decision Tree")
                    st.markdown(tree_analysis)

# ---------------------------------------------------------
# TAB 4: SYSTEM DECISION TREES (Static HTML / Markdown)
# ---------------------------------------------------------
with tab4:
    st.header("🌳 Core Grappling Decision Trees")

    trees_html = """
    <div style="background-color: #1e293b; color: #f8fafc; padding: 20px; border-radius: 8px; font-family: monospace;">
        <h3 style="color: #38bdf8; margin-top:0;">1. Kimura Reaction Chain (Adele Fornarino System)</h3>
        <pre style="color: #f8fafc; font-size: 14px;">
KIMURA GRIP ESTABLISHED
   │
   ▼ (Opponent pushes back)
Posture Break ➔ Collar Tie ➔ Shrimp ➔ Foot on Hip ➔ ARMBAR ENTRY
   │
   ├──► [They Yank Arm Out] ➔ Catch in Clamp Guard ➔ FRONT TRIANGLE
   │
   └──► [They Duck Hand Under Hip] ➔ Frame Head + Tricep Grip
                                   ➔ Foot Over Head ➔ Hip Heist + Whizzer
                                   ➔ OMOPLATA FINISH
        </pre>
        <hr style="border-color: #475569;">
        <h3 style="color: #38bdf8;">2. Half Guard Rescue & Clamp Entry (Bodoni / Glick System)</h3>
        <pre style="color: #f8fafc; font-size: 14px;">
FLAT IN HALF GUARD (Opponent has Underhook)
   │
   ▼ Intercept cross-face with BICEP BLOCK + Head Back
   │
   ├──► [Is Knee Lever Available? YES] ➔ Knee-Elbow Escape x2 ➔ CLAMP GUARD
   │
   └──► [Is Knee Lever Available? NO]  ➔ Explosive Bridge + Windmill Sweep
                                       ➔ Block Knee + Palm-Up Wrist Grip
                                       ➔ Roll Frame Inside ➔ CLAMP GUARD
   │
   ▼ READ CLAMP REACTION
   ├──► Opponent Drives Forward ➔ Knee block frame ➔ Dorsiflexion push
   ├──► Opponent Postures / Pulls ➔ My Plata / Half-Butterfly / Bait Triangle
   └──► Opponent Stays Still ➔ Maintain pressure & attack
        </pre>
    </div>
    """
    st.markdown(trees_html, unsafe_allow_html=True)

# ---------------------------------------------------------
# TAB 5: ANALYTICS DASHBOARD
# ---------------------------------------------------------
with tab5:
    st.header("📊 Training Analytics")
    df_logs = pd.read_csv(LOG_FILE)

    if df_logs.empty:
        st.info("No training logs found. Log a session in Tab 1 to unlock analytics.")
    else:
        df_logs['Date'] = pd.to_datetime(df_logs['Date'])

        # Metric Cards
        col_m1, col_m2, col_m3, col_m4 = st.columns(4)
        col_m1.metric("Total Hours", f"{df_logs['Duration_Min'].sum() / 60:.1f} hrs")
        col_m2.metric("Total Sessions", len(df_logs))
        col_m3.metric("Total Sparring Rounds", df_logs['Rounds'].sum())
        col_m4.metric("Current Rank", f"{df_logs.iloc[-1]['Rank']} ({df_logs.iloc[-1]['Stripes']}★)")

        # Plotly Monthly Training Volume
        df_monthly = df_logs.groupby(['Style', pd.Grouper(key='Date', freq='ME')])['Duration_Min'].sum().reset_index()
        fig = px.bar(
            df_monthly, x='Date', y='Duration_Min', color='Style',
            title='Monthly Mat Volume (Minutes)',
            labels={'Duration_Min': 'Minutes', 'Date': 'Month'},
            barmode='stack'
        )
        st.plotly_chart(fig, use_container_width=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 14.9 MB/s eta 0:00:00


2026-09-27 14:05:05.681 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-27 14:05:05.696 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-27 14:05:05.864 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-09-27 14:05:05.865 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-27 14:05:05.866 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-27 14:05:05.869 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-27 14:05:05.871 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn